# Chapter 32
## M-Current PING and Poisson PING
- Code by : [Abolfazl Ziaeemehr](https://github.com/Ziaeemehr)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ITNG/ModelingNeuralDynamics/blob/main/python/chapter32.ipynb)

## About this chapter

This chapter compares two ways a PING (Pyramidal-Interneuron Network Gamma)
population can produce a *weak*, loosely-periodic rhythm instead of a
tightly-locked one: an M-current-mediated recovery in the E cells, and a
Poisson stream of independent excitatory events. In "M-current PING", each
E cell carries a slow, non-inactivating potassium current

$$
I_M = g_M\,w\,(v-E_K), \qquad \dot w = \frac{w_\infty(v)-w}{\tau_w(v)},
$$

whose slow buildup and decay make an E cell's next spike depend on how long
it has been since its own last spike, not only on the network's inhibitory
volley -- producing an irregular, cycle-skipping population rhythm. In
"Poisson PING" there is no M-current at all: each E cell instead receives
its own independent Poisson stream of excitatory synaptic events (mean rate
`f_stoch`, peak conductance `g_stoch`), and the population's gamma episodes
emerge from many uncorrelated individual events aligning through recurrent
inhibition rather than from a shared deterministic drive.

Interpreting *when* an E or I cell spikes within a cycle -- not just how
often -- is the job of the two phase maps `psi`/`phi` computed at the end of
this chapter: they describe how the phase of one spike maps to the phase of
the next under an inhibitory pulse, and are the tool `PING_CLUSTERS` uses to
explain temporal sub-clustering.

In [ ]:
import subprocess
import sys
if "google.colab" in sys.modules:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "modelingneuraldynamics"], check=True)

## RTM/WB neuron model and shared network helpers

Every example uses the same excitatory Reduced-Traub-Miles (RTM) cell and
inhibitory Wang-Buzsaki (WB) cell as earlier chapters, coupled through
double-exponential synapses. `m_e_inf`, `w_inf`, `tau_w`, etc. add the RTM
cell's M-current gating variable `w` (set `g_m=0` to recover a plain RTM
cell, as `simulate_ping_clusters` does below). `rtm_init_with_m_current_population`
and `wb_init_population` splay-initialize a whole population of uncoupled
cells around their limit cycle (or fixed point) at prescribed phases, so a
population simulation starts already "running". `simulate_m_current_ping_plain`
is the shared, plain-NumPy core reused (with different parameters, and
`g_m=0` where there is no M-current) by `M_CURRENT_PING_1` and
`PING_CLUSTERS`; the numba-accelerated sibling used by the other M-current
examples is defined further below.

In [ ]:
import numpy as np
from numpy import exp, tanh
import matplotlib.pyplot as plt
from ipywidgets import interact


# ------------------------------------------------------------- E cell (RTM + M-current)


def m_e_inf(v):
    alpha_m = 0.32 * (v + 54) / (1 - exp(-(v + 54) / 4))
    beta_m = 0.28 * (v + 27) / (exp((v + 27) / 5) - 1)
    return alpha_m / (alpha_m + beta_m)


def h_e_inf(v):
    alpha_h = 0.128 * exp(-(v + 50) / 18)
    beta_h = 4. / (1 + exp(-(v + 27) / 5))
    return alpha_h / (alpha_h + beta_h)


def tau_h_e(v):
    alpha_h = 0.128 * exp(-(v + 50) / 18)
    beta_h = 4. / (1 + exp(-(v + 27) / 5))
    return 1. / (alpha_h + beta_h)


def n_e_inf(v):
    alpha_n = 0.032 * (v + 52) / (1 - exp(-(v + 52) / 5))
    beta_n = 0.5 * exp(-(v + 57) / 40)
    return alpha_n / (alpha_n + beta_n)


def tau_n_e(v):
    alpha_n = 0.032 * (v + 52) / (1 - exp(-(v + 52) / 5))
    beta_n = 0.5 * exp(-(v + 57) / 40)
    return 1. / (alpha_n + beta_n)


def w_inf(v):
    return 1. / (1 + exp(-(v + 35) / 10))


def tau_w(v):
    return 400. / (3.3 * exp((v + 35) / 20) + exp(-(v + 35) / 20))


# ------------------------------------------------------------- I cell (WB)


def m_i_inf(v):
    alpha_m = 0.1 * (v + 35) / (1 - exp(-(v + 35) / 10))
    beta_m = 4. * exp(-(v + 60) / 18)
    return alpha_m / (alpha_m + beta_m)


def h_i_inf(v):
    alpha_h = 0.07 * exp(-(v + 58) / 20)
    beta_h = 1. / (exp(-0.1 * (v + 28)) + 1)
    return alpha_h / (alpha_h + beta_h)


def tau_h_i(v):
    alpha_h = 0.07 * exp(-(v + 58) / 20)
    beta_h = 1. / (exp(-0.1 * (v + 28)) + 1)
    return 1. / (alpha_h + beta_h) / 5


def n_i_inf(v):
    alpha_n = -0.01 * (v + 34) / (exp(-0.1 * (v + 34)) - 1)
    beta_n = 0.125 * exp(-(v + 44) / 80)
    return alpha_n / (alpha_n + beta_n)


def tau_n_i(v):
    alpha_n = -0.01 * (v + 34) / (exp(-0.1 * (v + 34)) - 1)
    beta_n = 0.125 * exp(-(v + 44) / 80)
    return 1. / (alpha_n + beta_n) / 5


# --------------------------------------------------------- double-exp synapse


def tau_peak_function(tau_d, tau_r, tau_d_q):
    dt_ = 0.01
    dt05_ = dt_ / 2
    s, t = 0., 0.
    s_inc = exp(-t / tau_d_q) * (1 - s) / tau_r - s * tau_d
    while s_inc > 0:
        t_old, s_inc_old = t, s_inc
        s_tmp = s + dt05_ * s_inc
        s_inc_tmp = exp(-(t + dt05_) / tau_d_q) * (1 - s_tmp) / tau_r - s_tmp / tau_d
        s = s + dt_ * s_inc_tmp
        t = t + dt_
        s_inc = exp(-t / tau_d_q) * (1 - s) / tau_r - s / tau_d
    return (t_old * (-s_inc) + t * s_inc_old) / (s_inc_old - s_inc)


def tau_d_q_function(tau_d, tau_r, tau_hat):
    tau_d_q_left = 1.
    while tau_peak_function(tau_d, tau_r, tau_d_q_left) > tau_hat:
        tau_d_q_left /= 2
    tau_d_q_right = tau_r
    while tau_peak_function(tau_d, tau_r, tau_d_q_right) < tau_hat:
        tau_d_q_right *= 2
    while tau_d_q_right - tau_d_q_left > 1e-12:
        tau_d_q_mid = (tau_d_q_left + tau_d_q_right) / 2
        if tau_peak_function(tau_d, tau_r, tau_d_q_mid) <= tau_hat:
            tau_d_q_left = tau_d_q_mid
        else:
            tau_d_q_right = tau_d_q_mid
    return (tau_d_q_left + tau_d_q_right) / 2


# ------------------------------------------------------- population splay init


def rtm_init_with_m_current_population(i_ext, phi_vec, g_m):
    '''vectorized RTM-with-M-current init over a population: each of
    len(i_ext) neurons is integrated (Heun/midpoint) independently until
    its 3rd spike, then (v,h,n,w) is interpolated at phase phi_vec[i]
    between the 2nd and 3rd spikes. Faithfully reproduces the matlab
    source\'s bug: m_tmp is computed from the pre-half-step v, not v_tmp.
    With g_m=0 this degenerates to a plain RTM population init (used by
    simulate_ping_clusters, which has no M-current).'''
    num = len(i_ext)
    max_spikes = 3
    t_final_init = 2000.
    dt_ = 0.01
    dt05_ = dt_ / 2

    v = -70. * np.ones(num)
    m = m_e_inf(v)
    h = h_e_inf(v)
    n = n_e_inf(v)
    w = np.zeros(num)
    t = 0.

    num_spikes = np.zeros(num, dtype=int)
    done = np.zeros(num, dtype=bool)
    t_spikes = np.zeros((num, max_spikes))
    out = np.zeros((num, 4))

    c, g_k, g_na, g_l = 1., 80., 100., 0.1
    v_k, v_na, v_l = -100., 50., -67.

    while np.sum(done) < num and t < t_final_init:
        v_old, h_old, n_old, w_old, t_old = v, h, n, w, t

        v_inc = (g_k * n ** 4 * (v_k - v) + g_na * m ** 3 * h * (v_na - v)
                  + g_l * (v_l - v) + g_m * w * (v_k - v) + i_ext) / c
        h_inc = (h_e_inf(v) - h) / tau_h_e(v)
        n_inc = (n_e_inf(v) - n) / tau_n_e(v)
        w_inc = (w_inf(v) - w) / tau_w(v)

        v_tmp = v + dt05_ * v_inc
        m_tmp = m_e_inf(v)  # faithful port of matlab's bug (uses v, not v_tmp)
        h_tmp = h + dt05_ * h_inc
        n_tmp = n + dt05_ * n_inc
        w_tmp = w + dt05_ * w_inc

        v_inc = (g_k * n_tmp ** 4 * (v_k - v_tmp) + g_na * m_tmp ** 3 * h_tmp * (v_na - v_tmp)
                  + g_l * (v_l - v_tmp) + g_m * w_tmp * (v_k - v_tmp) + i_ext) / c
        h_inc = (h_e_inf(v_tmp) - h_tmp) / tau_h_e(v_tmp)
        n_inc = (n_e_inf(v_tmp) - n_tmp) / tau_n_e(v_tmp)
        w_inc = (w_inf(v_tmp) - w_tmp) / tau_w(v_tmp)

        v = v + dt_ * v_inc
        m = m_e_inf(v)
        h = h + dt_ * h_inc
        n = n + dt_ * n_inc
        w = w + dt_ * w_inc
        t = t + dt_

        ind = np.where((v_old >= -20) & (v < -20))[0]
        for k in ind:
            num_spikes[k] += 1
            if num_spikes[k] <= max_spikes:
                t_spikes[k, num_spikes[k] - 1] = (t_old * (-20 - v[k]) + t * (v_old[k] + 20)) / (v_old[k] - v[k])

        thr = t_spikes[:, max_spikes - 1] + phi_vec * (t_spikes[:, max_spikes - 1] - t_spikes[:, max_spikes - 2])
        ind = np.where((num_spikes == max_spikes) & (t > thr) & (t_old <= thr) & (~done))[0]
        for k in ind:
            out[k, 0] = (v_old[k] * (t - thr[k]) + v[k] * (thr[k] - t_old)) / dt_
            out[k, 1] = (h_old[k] * (t - thr[k]) + h[k] * (thr[k] - t_old)) / dt_
            out[k, 2] = (n_old[k] * (t - thr[k]) + n[k] * (thr[k] - t_old)) / dt_
            out[k, 3] = (w_old[k] * (t - thr[k]) + w[k] * (thr[k] - t_old)) / dt_
        done[ind] = True

    ind = np.where(~done)[0]
    out[ind, 0] = v[ind]
    out[ind, 1] = h[ind]
    out[ind, 2] = n[ind]
    out[ind, 3] = w[ind]
    return out


def wb_init_population(i_ext, phi_vec):
    '''same idea as rtm_init_with_m_current_population, for a population
    of WB (I-cell) neurons; used by simulate_poisson_ping (i_init="wb") to
    splay-initialize the I-cell population instead of starting every I-cell
    at rest.'''
    num = len(i_ext)
    max_spikes = 3
    t_final_init = 2000.
    dt_ = 0.01
    dt05_ = dt_ / 2

    v = -70. * np.ones(num)
    m = m_i_inf(v)
    h = h_i_inf(v)
    n = n_i_inf(v)
    t = 0.

    num_spikes = np.zeros(num, dtype=int)
    done = np.zeros(num, dtype=bool)
    t_spikes = np.zeros((num, max_spikes))
    out = np.zeros((num, 3))

    c, g_k, g_na, g_l = 1., 9., 35., 0.1
    v_k, v_na, v_l = -90., 55., -65.

    while np.sum(done) < num and t < t_final_init:
        v_old, h_old, n_old, t_old = v, h, n, t

        v_inc = (g_k * n ** 4 * (v_k - v) + g_na * m ** 3 * h * (v_na - v) + g_l * (v_l - v) + i_ext) / c
        h_inc = (h_i_inf(v) - h) / tau_h_i(v)
        n_inc = (n_i_inf(v) - n) / tau_n_i(v)

        v_tmp = v + dt05_ * v_inc
        m_tmp = m_i_inf(v)  # faithful port of matlab's bug (uses v, not v_tmp)
        h_tmp = h + dt05_ * h_inc
        n_tmp = n + dt05_ * n_inc

        v_inc = (g_k * n_tmp ** 4 * (v_k - v_tmp) + g_na * m_tmp ** 3 * h_tmp * (v_na - v_tmp)
                  + g_l * (v_l - v_tmp) + i_ext) / c
        h_inc = (h_i_inf(v_tmp) - h_tmp) / tau_h_i(v_tmp)
        n_inc = (n_i_inf(v_tmp) - n_tmp) / tau_n_i(v_tmp)

        v = v + dt_ * v_inc
        m = m_i_inf(v)
        h = h + dt_ * h_inc
        n = n + dt_ * n_inc
        t = t + dt_

        ind = np.where((v_old >= -20) & (v < -20))[0]
        for k in ind:
            num_spikes[k] += 1
            if num_spikes[k] <= max_spikes:
                t_spikes[k, num_spikes[k] - 1] = (t_old * (-20 - v[k]) + t * (v_old[k] + 20)) / (v_old[k] - v[k])

        thr = t_spikes[:, max_spikes - 1] + phi_vec * (t_spikes[:, max_spikes - 1] - t_spikes[:, max_spikes - 2])
        ind = np.where((num_spikes == max_spikes) & (t > thr) & (t_old <= thr) & (~done))[0]
        for k in ind:
            out[k, 0] = (v_old[k] * (t - thr[k]) + v[k] * (thr[k] - t_old)) / dt_
            out[k, 1] = (h_old[k] * (t - thr[k]) + h[k] * (thr[k] - t_old)) / dt_
            out[k, 2] = (n_old[k] * (t - thr[k]) + n[k] * (thr[k] - t_old)) / dt_
        done[ind] = True

    ind = np.where(~done)[0]
    out[ind, 0] = v[ind]
    out[ind, 1] = h[ind]
    out[ind, 2] = n[ind]
    return out


def make_random_connectivity(num_e, num_i, g_hat_ee, g_hat_ei, g_hat_ie, g_hat_ii,
                              p_ee, p_ei, p_ie, p_ii, rng):
    '''each possible synapse exists independently with probability p_XY;
    weights are normalized by the expected in-degree so the expected total
    conductance onto a cell is g_hat_XY regardless of N or p.'''
    u_ee = rng.random((num_e, num_e))
    u_ei = rng.random((num_e, num_i))
    u_ie = rng.random((num_i, num_e))
    u_ii = rng.random((num_i, num_i))
    g_ee = g_hat_ee * (u_ee < p_ee) / (num_e * p_ee) if p_ee > 0 else np.zeros((num_e, num_e))
    g_ei = g_hat_ei * (u_ei < p_ei) / (num_e * p_ei)
    g_ie = g_hat_ie * (u_ie < p_ie) / (num_i * p_ie)
    g_ii = g_hat_ii * (u_ii < p_ii) / (num_i * p_ii)
    return g_ee, g_ei, g_ie, g_ii


def plot_ping_raster(t_e, i_e, t_i, i_i, num_e, num_i, t_final, ax=None, title=""):
    '''raster shared by every population example: I cells below, E cells
    above a dashed separator.'''
    if ax is None:
        _, ax = plt.subplots(figsize=(8, 4))
    if len(t_i) > 0:
        ax.plot(t_i, i_i, '.b', markersize=2)
    if len(t_e) > 0:
        ax.plot(t_e, i_e + num_i, '.r', markersize=2)
    ax.plot([0, t_final], [num_i + 0.5, num_i + 0.5], '--k', linewidth=1)
    ax.set_yticks([num_i, num_e + num_i])
    ax.axis([0, t_final, 0, num_e + num_i + 1])
    ax.set_xlabel('$t$ [ms]')
    if title:
        ax.set_title(title)
    return ax


def simulate_m_current_ping_plain(num_e, num_i, i_ext_e, i_ext_i, g_ee, g_ei, g_ie, g_ii, g_m,
                                   v_rev_e=0., v_rev_i=-75.,
                                   tau_r_e=0.5, tau_peak_e=0.5, tau_d_e=3.,
                                   tau_r_i=0.5, tau_peak_i=0.5, tau_d_i=9.,
                                   t_final=1000., dt=0.01, rng=None, track_cell=3):
    '''plain-NumPy M-current PING network stepper -- kept unaccelerated
    on purpose as the numba step loop\'s reference implementation (see
    simulate_m_current_ping_numba below), and reused directly with g_m=0
    for simulate_ping_clusters, which has no M-current.'''
    if rng is None:
        rng = np.random.default_rng(63806)
    dt05 = dt / 2
    m_steps = round(t_final / dt)
    tau_dq_e = tau_d_q_function(tau_d_e, tau_r_e, tau_peak_e)
    tau_dq_i = tau_d_q_function(tau_d_i, tau_r_i, tau_peak_i)

    iv = rtm_init_with_m_current_population(i_ext_e, rng.random(num_e), g_m)
    v_e, h_e, n_e, w = iv[:, 0], iv[:, 1], iv[:, 2], iv[:, 3]
    m_e = m_e_inf(v_e)
    q_e, s_e = np.zeros(num_e), np.zeros(num_e)

    v_i = -75. * np.ones(num_i)
    m_i, h_i, n_i = m_i_inf(v_i), h_i_inf(v_i), n_i_inf(v_i)
    q_i, s_i = np.zeros(num_i), np.zeros(num_i)

    t_e_spikes, i_e_spikes = [], []
    t_i_spikes, i_i_spikes = [], []
    v_plot = np.zeros(m_steps)

    for k in range(1, m_steps + 1):
        v_e_inc = (0.1 * (-67 - v_e) + 80 * n_e ** 4 * (-100 - v_e) + 100 * m_e ** 3 * h_e * (50 - v_e)
                    + (g_ee.T @ s_e) * (v_rev_e - v_e) + (g_ie.T @ s_i) * (v_rev_i - v_e)
                    + g_m * w * (-100 - v_e) + i_ext_e)
        n_e_inc = (n_e_inf(v_e) - n_e) / tau_n_e(v_e)
        h_e_inc = (h_e_inf(v_e) - h_e) / tau_h_e(v_e)
        q_e_inc = (1 + tanh(v_e / 10)) / 2 * (1 - q_e) / 0.1 - q_e / tau_dq_e
        s_e_inc = q_e * (1 - s_e) / tau_r_e - s_e / tau_d_e
        w_inc = (w_inf(v_e) - w) / tau_w(v_e)

        v_i_inc = (0.1 * (-65 - v_i) + 9 * n_i ** 4 * (-90 - v_i) + 35 * m_i ** 3 * h_i * (55 - v_i)
                    + (g_ei.T @ s_e) * (v_rev_e - v_i) + (g_ii.T @ s_i) * (v_rev_i - v_i) + i_ext_i)
        n_i_inc = (n_i_inf(v_i) - n_i) / tau_n_i(v_i)
        h_i_inc = (h_i_inf(v_i) - h_i) / tau_h_i(v_i)
        q_i_inc = (1 + tanh(v_i / 10)) / 2 * (1 - q_i) / 0.1 - q_i / tau_dq_i
        s_i_inc = q_i * (1 - s_i) / tau_r_i - s_i / tau_d_i

        v_e_tmp = v_e + dt05 * v_e_inc
        n_e_tmp = n_e + dt05 * n_e_inc
        m_e_tmp = m_e_inf(v_e_tmp)
        h_e_tmp = h_e + dt05 * h_e_inc
        q_e_tmp = q_e + dt05 * q_e_inc
        s_e_tmp = s_e + dt05 * s_e_inc
        w_tmp = w + dt05 * w_inc

        v_i_tmp = v_i + dt05 * v_i_inc
        n_i_tmp = n_i + dt05 * n_i_inc
        m_i_tmp = m_i_inf(v_i_tmp)
        h_i_tmp = h_i + dt05 * h_i_inc
        q_i_tmp = q_i + dt05 * q_i_inc
        s_i_tmp = s_i + dt05 * s_i_inc

        v_e_inc = (0.1 * (-67 - v_e_tmp) + 80 * n_e_tmp ** 4 * (-100 - v_e_tmp)
                    + 100 * m_e_tmp ** 3 * h_e_tmp * (50 - v_e_tmp)
                    + (g_ee.T @ s_e_tmp) * (v_rev_e - v_e_tmp) + (g_ie.T @ s_i_tmp) * (v_rev_i - v_e_tmp)
                    + g_m * w_tmp * (-100 - v_e_tmp) + i_ext_e)
        n_e_inc = (n_e_inf(v_e_tmp) - n_e_tmp) / tau_n_e(v_e_tmp)
        h_e_inc = (h_e_inf(v_e_tmp) - h_e_tmp) / tau_h_e(v_e_tmp)
        q_e_inc = (1 + tanh(v_e_tmp / 10)) / 2 * (1 - q_e_tmp) / 0.1 - q_e_tmp / tau_dq_e
        s_e_inc = q_e_tmp * (1 - s_e_tmp) / tau_r_e - s_e_tmp / tau_d_e
        w_inc = (w_inf(v_e_tmp) - w_tmp) / tau_w(v_e_tmp)

        v_i_inc = (0.1 * (-65 - v_i_tmp) + 9 * n_i_tmp ** 4 * (-90 - v_i_tmp)
                    + 35 * m_i_tmp ** 3 * h_i_tmp * (55 - v_i_tmp)
                    + (g_ei.T @ s_e_tmp) * (v_rev_e - v_i_tmp) + (g_ii.T @ s_i_tmp) * (v_rev_i - v_i_tmp) + i_ext_i)
        n_i_inc = (n_i_inf(v_i_tmp) - n_i_tmp) / tau_n_i(v_i_tmp)
        h_i_inc = (h_i_inf(v_i_tmp) - h_i_tmp) / tau_h_i(v_i_tmp)
        q_i_inc = (1 + tanh(v_i_tmp / 10)) / 2 * (1 - q_i_tmp) / 0.1 - q_i_tmp / tau_dq_i
        s_i_inc = q_i_tmp * (1 - s_i_tmp) / tau_r_i - s_i_tmp / tau_d_i

        v_e_old, v_i_old = v_e, v_i

        v_e = v_e + dt * v_e_inc
        m_e = m_e_inf(v_e)
        h_e = h_e + dt * h_e_inc
        n_e = n_e + dt * n_e_inc
        q_e = q_e + dt * q_e_inc
        s_e = s_e + dt * s_e_inc
        w = w + dt * w_inc

        v_i = v_i + dt * v_i_inc
        m_i = m_i_inf(v_i)
        h_i = h_i + dt * h_i_inc
        n_i = n_i + dt * n_i_inc
        q_i = q_i + dt * q_i_inc
        s_i = s_i + dt * s_i_inc

        which_e = np.where((v_e_old > -20) & (v_e <= -20))[0]
        which_i = np.where((v_i_old > -20) & (v_i <= -20))[0]
        if len(which_e) > 0:
            i_e_spikes.extend(which_e.tolist())
            t_e_spikes.extend((((-20 - v_e[which_e]) * (k - 1) * dt + (v_e_old[which_e] + 20) * k * dt)
                                / (-v_e[which_e] + v_e_old[which_e])).tolist())
        if len(which_i) > 0:
            i_i_spikes.extend(which_i.tolist())
            t_i_spikes.extend((((-20 - v_i[which_i]) * (k - 1) * dt + (v_i_old[which_i] + 20) * k * dt)
                                / (-v_i[which_i] + v_i_old[which_i])).tolist())

        v_plot[k - 1] = v_e[track_cell]

    return (np.array(t_e_spikes), np.array(i_e_spikes),
            np.array(t_i_spikes), np.array(i_i_spikes), v_plot)

## Numba-accelerated M-current network stepper

`_m_current_run_loop` is the compiled core reused by every M-current
example below except `M_CURRENT_PING_1` (which stays on the plain-NumPy
`simulate_m_current_ping_plain` above, so the two can be checked against
each other). Following this repo's numba convention, the per-step update
uses `math.pow(x, 3.0)`/`math.pow(x, 4.0)` instead of `x ** 3`/`x ** 4`
(numba's `**` differs from CPython's by about 1 ULP, which would otherwise
compound over a long integration), and the function is *not* decorated with
`@njit(cache=True)`. `_m_current_settle_loop` is a bookkeeping-free sibling
used only for `M_CURRENT_PING_1_FROM_REST`'s silent 200 ms settling phase.
Both were checked against `simulate_m_current_ping_plain` at reduced scale
(same parameters, `t_final=50`) before trusting them: population firing
rates and raster shape matched closely, with only the expected sub-percent
drift from the `math.pow` vs. `**` ULP difference.

In [ ]:
import math
from numba import njit
from numba.typed import List


@njit
def _m_e_inf_s(v):
    alpha_m = 0.32 * (v + 54) / (1 - math.exp(-(v + 54) / 4))
    beta_m = 0.28 * (v + 27) / (math.exp((v + 27) / 5) - 1)
    return alpha_m / (alpha_m + beta_m)


@njit
def _h_e_inf_s(v):
    alpha_h = 0.128 * math.exp(-(v + 50) / 18)
    beta_h = 4. / (1 + math.exp(-(v + 27) / 5))
    return alpha_h / (alpha_h + beta_h)


@njit
def _tau_h_e_s(v):
    alpha_h = 0.128 * math.exp(-(v + 50) / 18)
    beta_h = 4. / (1 + math.exp(-(v + 27) / 5))
    return 1. / (alpha_h + beta_h)


@njit
def _n_e_inf_s(v):
    alpha_n = 0.032 * (v + 52) / (1 - math.exp(-(v + 52) / 5))
    beta_n = 0.5 * math.exp(-(v + 57) / 40)
    return alpha_n / (alpha_n + beta_n)


@njit
def _tau_n_e_s(v):
    alpha_n = 0.032 * (v + 52) / (1 - math.exp(-(v + 52) / 5))
    beta_n = 0.5 * math.exp(-(v + 57) / 40)
    return 1. / (alpha_n + beta_n)


@njit
def _w_inf_s(v):
    return 1. / (1 + math.exp(-(v + 35) / 10))


@njit
def _tau_w_s(v):
    return 400. / (3.3 * math.exp((v + 35) / 20) + math.exp(-(v + 35) / 20))


@njit
def _m_i_inf_s(v):
    alpha_m = 0.1 * (v + 35) / (1 - math.exp(-(v + 35) / 10))
    beta_m = 4. * math.exp(-(v + 60) / 18)
    return alpha_m / (alpha_m + beta_m)


@njit
def _h_i_inf_s(v):
    alpha_h = 0.07 * math.exp(-(v + 58) / 20)
    beta_h = 1. / (math.exp(-0.1 * (v + 28)) + 1)
    return alpha_h / (alpha_h + beta_h)


@njit
def _tau_h_i_s(v):
    alpha_h = 0.07 * math.exp(-(v + 58) / 20)
    beta_h = 1. / (math.exp(-0.1 * (v + 28)) + 1)
    return 1. / (alpha_h + beta_h) / 5.


@njit
def _n_i_inf_s(v):
    alpha_n = -0.01 * (v + 34) / (math.exp(-0.1 * (v + 34)) - 1)
    beta_n = 0.125 * math.exp(-(v + 44) / 80)
    return alpha_n / (alpha_n + beta_n)


@njit
def _tau_n_i_s(v):
    alpha_n = -0.01 * (v + 34) / (math.exp(-0.1 * (v + 34)) - 1)
    beta_n = 0.125 * math.exp(-(v + 44) / 80)
    return 1. / (alpha_n + beta_n) / 5.


@njit
def _m_current_matvecs(num_e, num_i, g_ee, g_ei, g_ie, g_ii, s_e, s_i,
                        ee_term, ie_term, ei_term, ii_term):
    for i in range(num_e):
        acc = 0.0
        for j in range(num_e):
            acc += g_ee[j, i] * s_e[j]
        ee_term[i] = acc
        acc = 0.0
        for j in range(num_i):
            acc += g_ie[j, i] * s_i[j]
        ie_term[i] = acc
    for i in range(num_i):
        acc = 0.0
        for j in range(num_e):
            acc += g_ei[j, i] * s_e[j]
        ei_term[i] = acc
        acc = 0.0
        for j in range(num_i):
            acc += g_ii[j, i] * s_i[j]
        ii_term[i] = acc


@njit
def _m_current_settle_loop(m_steps, dt, dt05, num_e, num_i, g_m,
                            v_rev_e, v_rev_i, tau_r_e, tau_d_e, tau_dq_e,
                            tau_r_i, tau_d_i, tau_dq_i,
                            i_ext_e, i_ext_i, g_ee, g_ei, g_ie, g_ii,
                            v_e, h_e, n_e, m_e, q_e, s_e, w,
                            v_i, h_i, n_i, m_i, q_i, s_i):
    dve = np.empty(num_e); dne = np.empty(num_e); dhe = np.empty(num_e)
    dqe = np.empty(num_e); dse = np.empty(num_e); dw = np.empty(num_e)
    dvi = np.empty(num_i); dni = np.empty(num_i); dhi = np.empty(num_i)
    dqi = np.empty(num_i); dsi = np.empty(num_i)

    ve_m = np.empty(num_e); ne_m = np.empty(num_e); me_m = np.empty(num_e)
    he_m = np.empty(num_e); qe_m = np.empty(num_e); se_m = np.empty(num_e); w_m = np.empty(num_e)
    vi_m = np.empty(num_i); ni_m = np.empty(num_i); mi_m = np.empty(num_i)
    hi_m = np.empty(num_i); qi_m = np.empty(num_i); si_m = np.empty(num_i)

    ee_term = np.empty(num_e); ie_term = np.empty(num_e)
    ei_term = np.empty(num_i); ii_term = np.empty(num_i)

    for step in range(m_steps):
        _m_current_matvecs(num_e, num_i, g_ee, g_ei, g_ie, g_ii, s_e, s_i,
                            ee_term, ie_term, ei_term, ii_term)

        for j in range(num_e):
            v = v_e[j]
            dve[j] = (0.1 * (-67 - v) + 80 * math.pow(n_e[j], 4.0) * (-100 - v)
                      + 100 * math.pow(m_e[j], 3.0) * h_e[j] * (50 - v)
                      + ee_term[j] * (v_rev_e - v) + ie_term[j] * (v_rev_i - v)
                      + g_m * w[j] * (-100 - v) + i_ext_e[j])
            dne[j] = (_n_e_inf_s(v) - n_e[j]) / _tau_n_e_s(v)
            dhe[j] = (_h_e_inf_s(v) - h_e[j]) / _tau_h_e_s(v)
            th = math.tanh(v / 10)
            dqe[j] = (1 + th) / 2 * (1 - q_e[j]) / 0.1 - q_e[j] / tau_dq_e
            dse[j] = q_e[j] * (1 - s_e[j]) / tau_r_e - s_e[j] / tau_d_e
            dw[j] = (_w_inf_s(v) - w[j]) / _tau_w_s(v)
        for j in range(num_i):
            v = v_i[j]
            dvi[j] = (0.1 * (-65 - v) + 9 * math.pow(n_i[j], 4.0) * (-90 - v)
                      + 35 * math.pow(m_i[j], 3.0) * h_i[j] * (55 - v)
                      + ei_term[j] * (v_rev_e - v) + ii_term[j] * (v_rev_i - v) + i_ext_i[j])
            dni[j] = (_n_i_inf_s(v) - n_i[j]) / _tau_n_i_s(v)
            dhi[j] = (_h_i_inf_s(v) - h_i[j]) / _tau_h_i_s(v)
            th = math.tanh(v / 10)
            dqi[j] = (1 + th) / 2 * (1 - q_i[j]) / 0.1 - q_i[j] / tau_dq_i
            dsi[j] = q_i[j] * (1 - s_i[j]) / tau_r_i - s_i[j] / tau_d_i

        for j in range(num_e):
            ve_m[j] = v_e[j] + dt05 * dve[j]
            ne_m[j] = n_e[j] + dt05 * dne[j]
            me_m[j] = _m_e_inf_s(ve_m[j])
            he_m[j] = h_e[j] + dt05 * dhe[j]
            qe_m[j] = q_e[j] + dt05 * dqe[j]
            se_m[j] = s_e[j] + dt05 * dse[j]
            w_m[j] = w[j] + dt05 * dw[j]
        for j in range(num_i):
            vi_m[j] = v_i[j] + dt05 * dvi[j]
            ni_m[j] = n_i[j] + dt05 * dni[j]
            mi_m[j] = _m_i_inf_s(vi_m[j])
            hi_m[j] = h_i[j] + dt05 * dhi[j]
            qi_m[j] = q_i[j] + dt05 * dqi[j]
            si_m[j] = s_i[j] + dt05 * dsi[j]

        _m_current_matvecs(num_e, num_i, g_ee, g_ei, g_ie, g_ii, se_m, si_m,
                            ee_term, ie_term, ei_term, ii_term)

        for j in range(num_e):
            v = ve_m[j]
            dve[j] = (0.1 * (-67 - v) + 80 * math.pow(ne_m[j], 4.0) * (-100 - v)
                      + 100 * math.pow(me_m[j], 3.0) * he_m[j] * (50 - v)
                      + ee_term[j] * (v_rev_e - v) + ie_term[j] * (v_rev_i - v)
                      + g_m * w_m[j] * (-100 - v) + i_ext_e[j])
            dne[j] = (_n_e_inf_s(v) - ne_m[j]) / _tau_n_e_s(v)
            dhe[j] = (_h_e_inf_s(v) - he_m[j]) / _tau_h_e_s(v)
            th = math.tanh(v / 10)
            dqe[j] = (1 + th) / 2 * (1 - qe_m[j]) / 0.1 - qe_m[j] / tau_dq_e
            dse[j] = qe_m[j] * (1 - se_m[j]) / tau_r_e - se_m[j] / tau_d_e
            dw[j] = (_w_inf_s(v) - w_m[j]) / _tau_w_s(v)
        for j in range(num_i):
            v = vi_m[j]
            dvi[j] = (0.1 * (-65 - v) + 9 * math.pow(ni_m[j], 4.0) * (-90 - v)
                      + 35 * math.pow(mi_m[j], 3.0) * hi_m[j] * (55 - v)
                      + ei_term[j] * (v_rev_e - v) + ii_term[j] * (v_rev_i - v) + i_ext_i[j])
            dni[j] = (_n_i_inf_s(v) - ni_m[j]) / _tau_n_i_s(v)
            dhi[j] = (_h_i_inf_s(v) - hi_m[j]) / _tau_h_i_s(v)
            th = math.tanh(v / 10)
            dqi[j] = (1 + th) / 2 * (1 - qi_m[j]) / 0.1 - qi_m[j] / tau_dq_i
            dsi[j] = qi_m[j] * (1 - si_m[j]) / tau_r_i - si_m[j] / tau_d_i

        for j in range(num_e):
            v_e[j] = v_e[j] + dt * dve[j]
            m_e[j] = _m_e_inf_s(v_e[j])
            h_e[j] = h_e[j] + dt * dhe[j]
            n_e[j] = n_e[j] + dt * dne[j]
            q_e[j] = q_e[j] + dt * dqe[j]
            s_e[j] = s_e[j] + dt * dse[j]
            w[j] = w[j] + dt * dw[j]
        for j in range(num_i):
            v_i[j] = v_i[j] + dt * dvi[j]
            m_i[j] = _m_i_inf_s(v_i[j])
            h_i[j] = h_i[j] + dt * dhi[j]
            n_i[j] = n_i[j] + dt * dni[j]
            q_i[j] = q_i[j] + dt * dqi[j]
            s_i[j] = s_i[j] + dt * dsi[j]


@njit
def _m_current_run_loop(m_steps, dt, dt05, num_e, num_i, g_m,
                         v_rev_e, v_rev_i, tau_r_e, tau_d_e, tau_dq_e,
                         tau_r_i, tau_d_i, tau_dq_i,
                         i_ext_e, i_ext_i, g_ee, g_ei, g_ie, g_ii,
                         v_e, h_e, n_e, m_e, q_e, s_e, w,
                         v_i, h_i, n_i, m_i, q_i, s_i, v_plot, track_cell):
    e_times = List.empty_list(np.float64)
    e_indices = List.empty_list(np.int64)
    i_times = List.empty_list(np.float64)
    i_indices = List.empty_list(np.int64)

    dve = np.empty(num_e); dne = np.empty(num_e); dhe = np.empty(num_e)
    dqe = np.empty(num_e); dse = np.empty(num_e); dw = np.empty(num_e)
    dvi = np.empty(num_i); dni = np.empty(num_i); dhi = np.empty(num_i)
    dqi = np.empty(num_i); dsi = np.empty(num_i)

    ve_m = np.empty(num_e); ne_m = np.empty(num_e); me_m = np.empty(num_e)
    he_m = np.empty(num_e); qe_m = np.empty(num_e); se_m = np.empty(num_e); w_m = np.empty(num_e)
    vi_m = np.empty(num_i); ni_m = np.empty(num_i); mi_m = np.empty(num_i)
    hi_m = np.empty(num_i); qi_m = np.empty(num_i); si_m = np.empty(num_i)

    ve_old = np.empty(num_e)
    vi_old = np.empty(num_i)
    ee_term = np.empty(num_e); ie_term = np.empty(num_e)
    ei_term = np.empty(num_i); ii_term = np.empty(num_i)

    for step in range(m_steps):
        k = step + 1

        _m_current_matvecs(num_e, num_i, g_ee, g_ei, g_ie, g_ii, s_e, s_i,
                            ee_term, ie_term, ei_term, ii_term)

        for j in range(num_e):
            v = v_e[j]
            dve[j] = (0.1 * (-67 - v) + 80 * math.pow(n_e[j], 4.0) * (-100 - v)
                      + 100 * math.pow(m_e[j], 3.0) * h_e[j] * (50 - v)
                      + ee_term[j] * (v_rev_e - v) + ie_term[j] * (v_rev_i - v)
                      + g_m * w[j] * (-100 - v) + i_ext_e[j])
            dne[j] = (_n_e_inf_s(v) - n_e[j]) / _tau_n_e_s(v)
            dhe[j] = (_h_e_inf_s(v) - h_e[j]) / _tau_h_e_s(v)
            th = math.tanh(v / 10)
            dqe[j] = (1 + th) / 2 * (1 - q_e[j]) / 0.1 - q_e[j] / tau_dq_e
            dse[j] = q_e[j] * (1 - s_e[j]) / tau_r_e - s_e[j] / tau_d_e
            dw[j] = (_w_inf_s(v) - w[j]) / _tau_w_s(v)
        for j in range(num_i):
            v = v_i[j]
            dvi[j] = (0.1 * (-65 - v) + 9 * math.pow(n_i[j], 4.0) * (-90 - v)
                      + 35 * math.pow(m_i[j], 3.0) * h_i[j] * (55 - v)
                      + ei_term[j] * (v_rev_e - v) + ii_term[j] * (v_rev_i - v) + i_ext_i[j])
            dni[j] = (_n_i_inf_s(v) - n_i[j]) / _tau_n_i_s(v)
            dhi[j] = (_h_i_inf_s(v) - h_i[j]) / _tau_h_i_s(v)
            th = math.tanh(v / 10)
            dqi[j] = (1 + th) / 2 * (1 - q_i[j]) / 0.1 - q_i[j] / tau_dq_i
            dsi[j] = q_i[j] * (1 - s_i[j]) / tau_r_i - s_i[j] / tau_d_i

        for j in range(num_e):
            ve_m[j] = v_e[j] + dt05 * dve[j]
            ne_m[j] = n_e[j] + dt05 * dne[j]
            me_m[j] = _m_e_inf_s(ve_m[j])
            he_m[j] = h_e[j] + dt05 * dhe[j]
            qe_m[j] = q_e[j] + dt05 * dqe[j]
            se_m[j] = s_e[j] + dt05 * dse[j]
            w_m[j] = w[j] + dt05 * dw[j]
        for j in range(num_i):
            vi_m[j] = v_i[j] + dt05 * dvi[j]
            ni_m[j] = n_i[j] + dt05 * dni[j]
            mi_m[j] = _m_i_inf_s(vi_m[j])
            hi_m[j] = h_i[j] + dt05 * dhi[j]
            qi_m[j] = q_i[j] + dt05 * dqi[j]
            si_m[j] = s_i[j] + dt05 * dsi[j]

        _m_current_matvecs(num_e, num_i, g_ee, g_ei, g_ie, g_ii, se_m, si_m,
                            ee_term, ie_term, ei_term, ii_term)

        for j in range(num_e):
            v = ve_m[j]
            dve[j] = (0.1 * (-67 - v) + 80 * math.pow(ne_m[j], 4.0) * (-100 - v)
                      + 100 * math.pow(me_m[j], 3.0) * he_m[j] * (50 - v)
                      + ee_term[j] * (v_rev_e - v) + ie_term[j] * (v_rev_i - v)
                      + g_m * w_m[j] * (-100 - v) + i_ext_e[j])
            dne[j] = (_n_e_inf_s(v) - ne_m[j]) / _tau_n_e_s(v)
            dhe[j] = (_h_e_inf_s(v) - he_m[j]) / _tau_h_e_s(v)
            th = math.tanh(v / 10)
            dqe[j] = (1 + th) / 2 * (1 - qe_m[j]) / 0.1 - qe_m[j] / tau_dq_e
            dse[j] = qe_m[j] * (1 - se_m[j]) / tau_r_e - se_m[j] / tau_d_e
            dw[j] = (_w_inf_s(v) - w_m[j]) / _tau_w_s(v)
        for j in range(num_i):
            v = vi_m[j]
            dvi[j] = (0.1 * (-65 - v) + 9 * math.pow(ni_m[j], 4.0) * (-90 - v)
                      + 35 * math.pow(mi_m[j], 3.0) * hi_m[j] * (55 - v)
                      + ei_term[j] * (v_rev_e - v) + ii_term[j] * (v_rev_i - v) + i_ext_i[j])
            dni[j] = (_n_i_inf_s(v) - ni_m[j]) / _tau_n_i_s(v)
            dhi[j] = (_h_i_inf_s(v) - hi_m[j]) / _tau_h_i_s(v)
            th = math.tanh(v / 10)
            dqi[j] = (1 + th) / 2 * (1 - qi_m[j]) / 0.1 - qi_m[j] / tau_dq_i
            dsi[j] = qi_m[j] * (1 - si_m[j]) / tau_r_i - si_m[j] / tau_d_i

        for j in range(num_e):
            ve_old[j] = v_e[j]
        for j in range(num_i):
            vi_old[j] = v_i[j]

        for j in range(num_e):
            v_e[j] = v_e[j] + dt * dve[j]
            m_e[j] = _m_e_inf_s(v_e[j])
            h_e[j] = h_e[j] + dt * dhe[j]
            n_e[j] = n_e[j] + dt * dne[j]
            q_e[j] = q_e[j] + dt * dqe[j]
            s_e[j] = s_e[j] + dt * dse[j]
            w[j] = w[j] + dt * dw[j]
        for j in range(num_i):
            v_i[j] = v_i[j] + dt * dvi[j]
            m_i[j] = _m_i_inf_s(v_i[j])
            h_i[j] = h_i[j] + dt * dhi[j]
            n_i[j] = n_i[j] + dt * dni[j]
            q_i[j] = q_i[j] + dt * dqi[j]
            s_i[j] = s_i[j] + dt * dsi[j]

        for j in range(num_e):
            if ve_old[j] > -20 and v_e[j] <= -20:
                e_indices.append(j)
                e_times.append(((-20 - v_e[j]) * step * dt + (ve_old[j] + 20) * k * dt)
                                / (ve_old[j] - v_e[j]))
        for j in range(num_i):
            if vi_old[j] > -20 and v_i[j] <= -20:
                i_indices.append(j)
                i_times.append(((-20 - v_i[j]) * step * dt + (vi_old[j] + 20) * k * dt)
                                / (vi_old[j] - v_i[j]))

        v_plot[step] = v_e[track_cell]

    return e_times, e_indices, i_times, i_indices


def simulate_m_current_ping_numba(num_e, num_i, i_ext_e, i_ext_i, g_ee, g_ei, g_ie, g_ii, g_m,
                                   v_rev_e=0., v_rev_i=-75.,
                                   tau_r_e=0.5, tau_peak_e=0.5, tau_d_e=3.,
                                   tau_r_i=0.5, tau_peak_i=0.5, tau_d_i=9.,
                                   t_final=1000., dt=0.01, rng=None, track_cell=3):
    if rng is None:
        rng = np.random.default_rng(63806)
    dt05 = dt / 2
    m_steps = round(t_final / dt)
    tau_dq_e = tau_d_q_function(tau_d_e, tau_r_e, tau_peak_e)
    tau_dq_i = tau_d_q_function(tau_d_i, tau_r_i, tau_peak_i)

    iv = rtm_init_with_m_current_population(i_ext_e, rng.random(num_e), g_m)
    v_e, h_e, n_e, w = iv[:, 0], iv[:, 1], iv[:, 2], iv[:, 3]
    m_e = m_e_inf(v_e)
    q_e, s_e = np.zeros(num_e), np.zeros(num_e)

    v_i = -75. * np.ones(num_i)
    m_i, h_i, n_i = m_i_inf(v_i), h_i_inf(v_i), n_i_inf(v_i)
    q_i, s_i = np.zeros(num_i), np.zeros(num_i)

    v_plot = np.zeros(m_steps)

    t_e_spikes, i_e_spikes, t_i_spikes, i_i_spikes = _m_current_run_loop(
        m_steps, dt, dt05, num_e, num_i, g_m,
        v_rev_e, v_rev_i, tau_r_e, tau_d_e, tau_dq_e,
        tau_r_i, tau_d_i, tau_dq_i,
        i_ext_e, i_ext_i, g_ee, g_ei, g_ie, g_ii,
        v_e, h_e, n_e, m_e, q_e, s_e, w,
        v_i, h_i, n_i, m_i, q_i, s_i, v_plot, track_cell,
    )
    t_e_spikes = np.array(t_e_spikes) if len(t_e_spikes) else np.empty(0)
    i_e_spikes = np.array(i_e_spikes, dtype=int) if len(i_e_spikes) else np.empty(0, dtype=int)
    t_i_spikes = np.array(t_i_spikes) if len(t_i_spikes) else np.empty(0)
    i_i_spikes = np.array(i_i_spikes, dtype=int) if len(i_i_spikes) else np.empty(0, dtype=int)
    return t_e_spikes, i_e_spikes, t_i_spikes, i_i_spikes, v_plot

## M_CURRENT_PING_1: baseline M-current weak PING

200 E cells (RTM + M-current, `g_m=1`) and 50 I cells (WB), sparsely
(`p=0.5`) and weakly (`g_hat_ei=g_hat_ie=g_hat_ii=0.5`) coupled, driven by
heterogeneous constant currents. `simulate_m_current_ping_1` is the
plain-NumPy reference simulation: it draws from a fresh `np.random.default_rng(seed)`
internally, so every call is independently reproducible (not just the
first).

In [ ]:
def simulate_m_current_ping_1(t_final_run=1000., seed=63806, track_cell=3, g_m=1.0):
    num_e, num_i = 200, 50
    rng = np.random.default_rng(seed)
    sigma_e = sigma_i = 0.05
    i_ext_e = 3.0 * np.ones(num_e) * (1 + sigma_e * rng.standard_normal(num_e))
    i_ext_i = 0.7 * np.ones(num_i) * (1 + sigma_i * rng.standard_normal(num_i))
    g_ee, g_ei, g_ie, g_ii = make_random_connectivity(
        num_e, num_i, 0., 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, rng)
    return simulate_m_current_ping_plain(
        num_e, num_i, i_ext_e, i_ext_i, g_ee, g_ei, g_ie, g_ii, g_m,
        t_final=t_final_run, rng=rng, track_cell=track_cell)

In [ ]:
t_e_spikes, i_e_spikes, t_i_spikes, i_i_spikes, v_plot = simulate_m_current_ping_1()
num_e, num_i, t_final, dt = 200, 50, 1000., 0.01
f_hat_e = round(len(t_e_spikes) / num_e / t_final * 1000)
f_hat_i = round(len(t_i_spikes) / num_i / t_final * 1000)
print(f"f_hat_e = {f_hat_e}, f_hat_i = {f_hat_i}")

fig, axes = plt.subplots(2, 1, figsize=(8, 6))
plot_ping_raster(t_e_spikes, i_e_spikes, t_i_spikes, i_i_spikes, num_e, num_i, t_final, ax=axes[0])
axes[1].plot(np.arange(1, len(v_plot) + 1) * dt, v_plot, '-k', linewidth=2)
axes[1].set_xlabel('$t$ [ms]')
axes[1].set_ylabel('$v$ [mV]')
plt.tight_layout()
plt.show()

The M-current conductance `g_m` sets how strongly each E cell's own
recent spiking suppresses its next spike -- explore it below (`g_m=0`
recovers an ordinary, tightly-locked PING rhythm).

In [ ]:
interact(lambda g_m=1.0: plot_ping_raster(
    *simulate_m_current_ping_1(t_final_run=300., g_m=g_m)[:4], 200, 50, 300.,
    title=f"g_m = {g_m:.2f}"), g_m=(0.0, 2.0, 0.1));

## M_CURRENT_PING_1_CLOSEUP, _2_CLOSEUP, _3_CLOSEUP: within-cycle timing

Same network and parameters as `M_CURRENT_PING_1`, run through the
numba-accelerated `simulate_m_current_ping_numba` and re-plotted zoomed in
on a 20-cell, 20 ms window to show the within-cycle E-cell recruitment
order that the M-current staggers. The legacy per-example scripts for
`_1_CLOSEUP`/`_2_CLOSEUP`/`_3_CLOSEUP` are identical, so all three closeups
below call the same function.

In [ ]:
def simulate_m_current_ping_closeup(t_final=1000., seed=63806, track_cell=3, g_m=1.0):
    num_e, num_i = 200, 50
    rng = np.random.default_rng(seed)
    sigma_e = sigma_i = 0.05
    i_ext_e = 3.0 * np.ones(num_e) * (1 + sigma_e * rng.standard_normal(num_e))
    i_ext_i = 0.7 * np.ones(num_i) * (1 + sigma_i * rng.standard_normal(num_i))
    g_ee, g_ei, g_ie, g_ii = make_random_connectivity(
        num_e, num_i, 0., 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, rng)
    return simulate_m_current_ping_numba(
        num_e, num_i, i_ext_e, i_ext_i, g_ee, g_ei, g_ie, g_ii, g_m,
        t_final=t_final, rng=rng, track_cell=track_cell)


def plot_m_current_closeup(t_e_spikes, i_e_spikes, t_final, num_i=50, title=""):
    '''closeup of E-cells 1-20 (plotted at 51-70), with local firing-rate
    maxima marked, following the matlab source\'s rastergram.m.'''
    dt = 0.01
    fig, ax = plt.subplots(figsize=(8, 5))
    e_mask = i_e_spikes < 20
    if e_mask.any():
        ax.plot(t_e_spikes[e_mask], i_e_spikes[e_mask] + num_i + 1, '.r', markersize=6)
    ax.set_yticks([51, 70])
    ax.axis([0, t_final, 50.5, 70.5])
    ax.set_xlabel('$t$ [ms]')
    if title:
        ax.set_title(title)

    m_steps = round(t_final / dt)
    t_vec_lfp = np.arange(m_steps + 1) * dt
    rate = np.zeros(m_steps + 1)
    sigma = 3.
    for tk in t_e_spikes:
        rate += np.exp(-(t_vec_lfp - tk) ** 2 / (2 * sigma ** 2))
    rate_c, rate_l, rate_r = rate[1:-1], rate[:-2], rate[2:]
    ind = np.where((rate_c > rate_l) & (rate_c > rate_r))[0]
    for tp in ind * dt:
        ax.plot([tp, tp], [50.5, 70.5], '-k', linewidth=1)
    return fig

In [ ]:
t_e_1, i_e_1, t_i_1, i_i_1, v_plot_1 = simulate_m_current_ping_closeup()
plot_m_current_closeup(t_e_1, i_e_1, 1000., title="M_CURRENT_PING_1_CLOSEUP")
plt.tight_layout()
plt.show()

## M_CURRENT_PING_1_FROM_REST: starting the network at rest

Identical network to `M_CURRENT_PING_1`, but instead of splay-initializing
each cell around its limit cycle, every cell starts at its resting
potential with zero drive for 200 ms (`_m_current_settle_loop`), and only
then is the network drive turned on for the usual 1000 ms
(`_m_current_run_loop`) -- so the raster below should stay empty for the
first stretch, until the network settles into its own rhythm.

In [ ]:
def simulate_m_current_ping_1_from_rest(t_final=1000., seed=63806, g_m=1.0):
    num_e, num_i = 200, 50
    dt, dt05 = 0.01, 0.005
    rng = np.random.default_rng(seed)
    sigma_e = sigma_i = 0.05
    i_ext_e_rest = np.zeros(num_e)
    i_ext_i_rest = np.zeros(num_i)

    tau_r_e, tau_peak_e, tau_d_e = 0.5, 0.5, 3.
    tau_r_i, tau_peak_i, tau_d_i = 0.5, 0.5, 9.
    tau_dq_e = tau_d_q_function(tau_d_e, tau_r_e, tau_peak_e)
    tau_dq_i = tau_d_q_function(tau_d_i, tau_r_i, tau_peak_i)

    g_ee, g_ei, g_ie, g_ii = make_random_connectivity(
        num_e, num_i, 0., 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, rng)

    v_e = -70. * np.ones(num_e)
    m_e, h_e, n_e = m_e_inf(v_e), h_e_inf(v_e), n_e_inf(v_e)
    q_e, s_e, w = np.zeros(num_e), np.zeros(num_e), np.zeros(num_e)

    v_i = -75. * np.ones(num_i)
    m_i, h_i, n_i = m_i_inf(v_i), h_i_inf(v_i), n_i_inf(v_i)
    q_i, s_i = np.zeros(num_i), np.zeros(num_i)

    m_steps_200 = round(200. / dt)
    _m_current_settle_loop(
        m_steps_200, dt, dt05, num_e, num_i, g_m,
        0., -75., tau_r_e, tau_d_e, tau_dq_e, tau_r_i, tau_d_i, tau_dq_i,
        i_ext_e_rest, i_ext_i_rest, g_ee, g_ei, g_ie, g_ii,
        v_e, h_e, n_e, m_e, q_e, s_e, w, v_i, h_i, n_i, m_i, q_i, s_i,
    )

    i_ext_e = 3.0 * np.ones(num_e) * (1 + sigma_e * rng.standard_normal(num_e))
    i_ext_i = 0.7 * np.ones(num_i) * (1 + sigma_i * rng.standard_normal(num_i))
    m_steps = round(t_final / dt)
    v_plot = np.zeros(m_steps)
    t_e_spikes, i_e_spikes, t_i_spikes, i_i_spikes = _m_current_run_loop(
        m_steps, dt, dt05, num_e, num_i, g_m,
        0., -75., tau_r_e, tau_d_e, tau_dq_e, tau_r_i, tau_d_i, tau_dq_i,
        i_ext_e, i_ext_i, g_ee, g_ei, g_ie, g_ii,
        v_e, h_e, n_e, m_e, q_e, s_e, w, v_i, h_i, n_i, m_i, q_i, s_i, v_plot, 3,
    )
    t_e_spikes = np.array(t_e_spikes) if len(t_e_spikes) else np.empty(0)
    i_e_spikes = np.array(i_e_spikes, dtype=int) if len(i_e_spikes) else np.empty(0, dtype=int)
    t_i_spikes = np.array(t_i_spikes) if len(t_i_spikes) else np.empty(0)
    i_i_spikes = np.array(i_i_spikes, dtype=int) if len(i_i_spikes) else np.empty(0, dtype=int)
    return t_e_spikes, i_e_spikes, t_i_spikes, i_i_spikes

In [ ]:
t_e_r, i_e_r, t_i_r, i_i_r = simulate_m_current_ping_1_from_rest()
plot_ping_raster(t_e_r, i_e_r, t_i_r, i_i_r, 200, 50, 1000., title="M_CURRENT_PING_1_FROM_REST")
plt.tight_layout()
plt.show()

## PING_CLUSTERS: temporal sub-clusters within a PING volley

An ordinary (no M-current, `g_m=0`) but strongly and densely coupled
(`p=1`, `g_hat_ei=5`, `g_hat_ie=0.25`, `g_hat_ii=0.75`) E-I network, run for
500 ms. `simulate_ping_clusters` reuses `simulate_m_current_ping_plain` with
`g_m=0`, which is mathematically identical to a plain-RTM population
(the `g_m * w * (...)` term vanishes regardless of `w`). Read this raster
alongside the `psi`/`phi` phase maps further below: temporal sub-clustering
within an E-cell volley is explained by where each cell's post-inhibition
phase falls on those maps.

In [ ]:
def simulate_ping_clusters(t_final=500., seed=63806):
    num_e, num_i = 200, 50
    rng = np.random.default_rng(seed)
    i_ext_e = 1.4 * np.ones(num_e)
    i_ext_i = 0.2 * np.ones(num_i)
    g_ee, g_ei, g_ie, g_ii = make_random_connectivity(
        num_e, num_i, 0., 5., 0.25, 0.75, 1., 1., 1., 1., rng)
    return simulate_m_current_ping_plain(
        num_e, num_i, i_ext_e, i_ext_i, g_ee, g_ei, g_ie, g_ii, g_m=0.0,
        tau_r_e=0.05, tau_peak_e=0.05, tau_d_e=1.,
        tau_r_i=0.1, tau_peak_i=0.1, tau_d_i=9.,
        t_final=t_final, rng=rng, track_cell=0)

In [ ]:
t_e_c, i_e_c, t_i_c, i_i_c, _ = simulate_ping_clusters()
fig, ax = plt.subplots(figsize=(8, 5))
if len(t_i_c) > 0:
    ax.plot(t_i_c, i_i_c + 1, '.b', markersize=6)
if len(t_e_c) > 0:
    ax.plot(t_e_c, i_e_c + 50, '.r', markersize=6)
ax.plot([0, 500.], [50.5, 50.5], '--k', linewidth=1)
ax.set_yticks([50, 70])
ax.axis([300., 500., 49.5, 70])
ax.set_xlabel('$t$ [ms]')
ax.set_title("PING_CLUSTERS")
plt.tight_layout()
plt.show()

## Phase maps `psi` and `phi`

An isolated LIF-like pair (membrane time constant `tau_m`, constant drive
`I`) coupled by a single inhibitory pulse of peak conductance `g_I` and
decay `tau_I` gives a simple reduction of "how does phase before an
inhibitory volley map to phase after it". `psif` starts the synaptic drive
`s` at 1 (a spike has just arrived) and lets it decay exponentially --
`simulate_plot_psi` calls it once to get `psi(x)`, the whole map;
`simulate_plot_phi` composes it with itself to get `phi(x) = psi(psi(x))`,
the two-inhibitory-pulses-ahead map used to argue that some fixed points
are stable. `psif_alpha`/`simulate_plot_psi_phi` instead use an alpha
function of absolute time for `s(t)` (matching the matlab source used for
that particular figure) rather than a spike-triggered reset.

In [ ]:
def psif(tau_m, I, g_I, tau_I, dt, x):
    '''x can be a 1-d array, in which case so is the output.'''
    v_1 = 0.
    v_2 = x.copy()
    s = 1.
    N = len(x)
    out = np.zeros(N)
    done = np.zeros(N, dtype=bool)

    dt05 = dt / 2

    while not done.all():
        v_1_old = v_1
        v_2_old = v_2.copy()
        v_1_inc = -v_1 / tau_m + I - g_I * s * v_1
        v_2_inc = -v_2 / tau_m + I - g_I * s * v_2
        v_1_tmp = v_1 + dt05 * v_1_inc
        v_2_tmp = v_2 + dt05 * v_2_inc
        s_tmp = s * np.exp(-dt05 / tau_I)
        v_1_inc = -v_1_tmp / tau_m + I - g_I * s_tmp * v_1_tmp
        v_2_inc = -v_2_tmp / tau_m + I - g_I * s_tmp * v_2_tmp
        v_1 = v_1 + dt * v_1_inc
        v_2 = v_2 + dt * v_2_inc

        ind = np.where((v_2 > 1) & ~done)[0]
        out[ind] = (v_1_old * (v_2[ind] - 1) + v_1 * (1 - v_2_old[ind])) / (v_2[ind] - v_2_old[ind])
        done[ind] = True
        s = s * np.exp(-dt / tau_I)

    return out


def simulate_plot_psi(tau_m=10., I=0.12, g_I=0.05, tau_I=5., dt=0.01, n=10000):
    x = np.arange(1, n) / n
    psi_vec = psif(tau_m, I, g_I, tau_I, dt, x)
    return x, psi_vec


def simulate_plot_phi(tau_m=10., I=0.12, g_I=0.05, tau_I=5., dt=0.01, n=10000):
    x, psi_vec = simulate_plot_psi(tau_m, I, g_I, tau_I, dt, n)
    phi_vec = psif(tau_m, I, g_I, tau_I, dt, psi_vec)
    return x, phi_vec


def psif_alpha(tau_m, I, g_I, tau_I, dt, x):
    '''like psif, but the synaptic drive s(t) is a fixed alpha function of
    absolute time (not reset/renormalized to 1 at t=0 with exponential decay
    tied to spike count), matching the matlab source.'''
    v_1 = 0.
    v_2 = x.copy()
    N = len(x)
    out = np.zeros(N)
    done = np.zeros(N, dtype=bool)

    delta = 0.

    def alpha(t):
        return np.exp(-(t - delta) / tau_I) * (t >= delta)

    dt05 = dt / 2
    t = 0.
    s = alpha(0.)

    while not done.all():
        v_1_old = v_1
        v_2_old = v_2.copy()
        v_1_inc = -v_1 / tau_m + I - g_I * s * v_1
        v_2_inc = -v_2 / tau_m + I - g_I * s * v_2
        v_1_tmp = v_1 + dt05 * v_1_inc
        v_2_tmp = v_2 + dt05 * v_2_inc
        s_tmp = alpha(t + dt05)
        v_1_inc = -v_1_tmp / tau_m + I - g_I * s_tmp * v_1_tmp
        v_2_inc = -v_2_tmp / tau_m + I - g_I * s_tmp * v_2_tmp
        v_1 = v_1 + dt * v_1_inc
        v_2 = v_2 + dt * v_2_inc

        ind = np.where((v_2 > 1) & ~done)[0]
        out[ind] = (v_1_old * (v_2[ind] - 1) + v_1 * (1 - v_2_old[ind])) / (v_2[ind] - v_2_old[ind])
        done[ind] = True
        t = t + dt
        s = alpha(t)

    return out


def simulate_plot_psi_phi(tau_m=10., I=0.12, g_I=0.015, tau_I=5., dt=0.01, n=10000):
    x = np.arange(1, n) / n
    psi_vec = psif_alpha(tau_m, I, g_I, tau_I, dt, x)
    phi_vec = psif_alpha(tau_m, I, g_I, tau_I, dt, psi_vec)
    return x, psi_vec, phi_vec

In [ ]:
x, phi_vec = simulate_plot_phi()
fig, ax = plt.subplots(figsize=(5, 5))
ax.plot(x, phi_vec, '-k', linewidth=4)
ax.plot([0, 1], [0, 1], '--k', linewidth=2)
ax.axis([0, 1, 0, 1])
ax.set_xlabel('$x$')
ax.set_ylabel(r'$\phi(x)$')
ax.set_aspect('equal')
plt.tight_layout()
plt.show()

In [ ]:
x, psi_vec = simulate_plot_psi()
print(f"max_psi = {psi_vec.max():.4f}, min_psi = {psi_vec.min():.4f}")
fig, ax = plt.subplots(figsize=(5, 5))
ax.plot(x, psi_vec, '-k', linewidth=4)
ax.plot([0, 1], [0, 1], '--k', linewidth=2)
ax.axis([0, 1, 0, 1])
ax.set_xlabel('$x$')
ax.set_ylabel(r'$\psi(x)$')
ax.set_aspect('equal')
plt.tight_layout()
plt.show()

In [ ]:
x, psi_vec, phi_vec = simulate_plot_psi_phi()
fig, axes = plt.subplots(1, 2, figsize=(8, 4))
axes[0].plot(x, psi_vec, '.k', markersize=2)
axes[0].plot([0, 1], [0, 1], '--k', linewidth=1)
axes[0].axis([0, 1, 0, 1]); axes[0].set_xlabel('$x$'); axes[0].set_ylabel(r'$\psi$')
axes[0].set_aspect('equal')
axes[1].plot(x, phi_vec, '.k', markersize=2)
axes[1].plot([0, 1], [0, 1], '--k', linewidth=1)
axes[1].axis([0, 1, 0, 1]); axes[1].set_xlabel('$x$'); axes[1].set_ylabel(r'$\phi$')
axes[1].set_aspect('equal')
plt.tight_layout()
plt.show()

`g_I` sets the strength of the single inhibitory pulse shaping the map;
larger `g_I` pushes `psi(x)` further from the diagonal (a stronger phase
reset).

In [ ]:
interact(lambda g_I=0.05: plt.plot(*simulate_plot_psi(g_I=g_I)[:2], '-k', linewidth=3)
         or plt.plot([0, 1], [0, 1], '--k') or plt.axis([0, 1, 0, 1])
         or plt.gca().set_aspect('equal') or plt.show(), g_I=(0.0, 0.2, 0.01));

## POISSON_PING_1, _2, _3: Poisson-driven weak PING

No M-current (`g_m=0`); instead each E cell gets its own independent
Poisson stream of excitatory synaptic events at mean rate `f_stoch` and
peak conductance `g_stoch`, on top of a double-exponential synapse tracked
by `q_stoch`/`s_stoch`. `simulate_poisson_ping` is the shared plain-NumPy
stepper (kept unaccelerated, like the legacy scripts, since the per-step
Poisson draw doesn't fit cleanly into a numba loop); `i_init="wb"`
splay-initializes the I-cell population instead of starting every I-cell at
rest (used once the I cells receive enough drive to be spiking on their
own). `POISSON_PING_1` has no I-cell drive at all (`i_ext_i_mean=0`);
`POISSON_PING_2` adds `i_ext_i_mean=0.8`; `POISSON_PING_3` uses much
stronger, all-to-all coupling with a faster synapse and a sparser/faster
Poisson stream, producing tightly synchronized I-cell volleys.

In [ ]:
def simulate_poisson_ping(num_e=200, num_i=50, i_ext_e_mean=0.5, sigma_e=0.05,
                           i_ext_i_mean=0.0, sigma_i=0.05,
                           g_hat_ee=0., g_hat_ei=0.25, g_hat_ie=0.25, g_hat_ii=0.25,
                           p_ee=0.5, p_ei=0.5, p_ie=0.5, p_ii=0.5,
                           tau_r_e=0.5, tau_peak_e=0.5, tau_d_e=3.,
                           tau_r_i=0.5, tau_peak_i=0.5, tau_d_i=9.,
                           f_stoch=60., g_stoch=0.03,
                           t_final=500., dt=0.01, seed=63806, i_init="constant", track_cell=2):
    v_rev_e, v_rev_i = 0., -75.
    rng = np.random.default_rng(seed)
    dt05 = dt / 2
    m_steps = round(t_final / dt)
    i_ext_e = i_ext_e_mean * np.ones(num_e) * (1 + sigma_e * rng.standard_normal(num_e))
    i_ext_i = i_ext_i_mean * np.ones(num_i) * (1 + sigma_i * rng.standard_normal(num_i))

    tau_dq_e = tau_d_q_function(tau_d_e, tau_r_e, tau_peak_e)
    tau_dq_i = tau_d_q_function(tau_d_i, tau_r_i, tau_peak_i)

    g_ee, g_ei, g_ie, g_ii = make_random_connectivity(
        num_e, num_i, g_hat_ee, g_hat_ei, g_hat_ie, g_hat_ii, p_ee, p_ei, p_ie, p_ii, rng)

    iv = rtm_init_with_m_current_population(i_ext_e, rng.random(num_e), 0.0)
    v_e, h_e, n_e = iv[:, 0], iv[:, 1], iv[:, 2]
    m_e = m_e_inf(v_e)
    q_e, s_e = np.zeros(num_e), np.zeros(num_e)
    q_stoch, s_stoch = np.zeros(num_e), np.zeros(num_e)

    if i_init == "wb":
        iv_i = wb_init_population(i_ext_i, rng.random(num_i))
        v_i, h_i, n_i = iv_i[:, 0], iv_i[:, 1], iv_i[:, 2]
        m_i = m_i_inf(v_i)
    else:
        v_i = -75. * np.ones(num_i)
        m_i, h_i, n_i = m_i_inf(v_i), h_i_inf(v_i), n_i_inf(v_i)
    q_i, s_i = np.zeros(num_i), np.zeros(num_i)

    t_e_spikes, i_e_spikes = [], []
    t_i_spikes, i_i_spikes = [], []
    v_one = np.zeros(m_steps + 1)
    v_one[0] = v_e[track_cell]

    for k in range(1, m_steps + 1):
        v_e_inc = (0.1 * (-67 - v_e) + 80 * n_e ** 4 * (-100 - v_e) + 100 * m_e ** 3 * h_e * (50 - v_e)
                    + (g_ee.T @ s_e) * (v_rev_e - v_e) + (g_ie.T @ s_i) * (v_rev_i - v_e)
                    + i_ext_e + g_stoch * s_stoch * (v_rev_e - v_e))
        n_e_inc = (n_e_inf(v_e) - n_e) / tau_n_e(v_e)
        h_e_inc = (h_e_inf(v_e) - h_e) / tau_h_e(v_e)
        q_e_inc = (1 + tanh(v_e / 10)) / 2 * (1 - q_e) / 0.1 - q_e / tau_dq_e
        s_e_inc = q_e * (1 - s_e) / tau_r_e - s_e / tau_d_e
        q_stoch_inc = -q_stoch / tau_dq_e
        s_stoch_inc = q_stoch * (1 - s_stoch) / tau_r_e - s_stoch / tau_d_e

        v_i_inc = (0.1 * (-65 - v_i) + 9 * n_i ** 4 * (-90 - v_i) + 35 * m_i ** 3 * h_i * (55 - v_i)
                    + (g_ei.T @ s_e) * (v_rev_e - v_i) + (g_ii.T @ s_i) * (v_rev_i - v_i) + i_ext_i)
        n_i_inc = (n_i_inf(v_i) - n_i) / tau_n_i(v_i)
        h_i_inc = (h_i_inf(v_i) - h_i) / tau_h_i(v_i)
        q_i_inc = (1 + tanh(v_i / 10)) / 2 * (1 - q_i) / 0.1 - q_i / tau_dq_i
        s_i_inc = q_i * (1 - s_i) / tau_r_i - s_i / tau_d_i

        v_e_tmp = v_e + dt05 * v_e_inc
        n_e_tmp = n_e + dt05 * n_e_inc
        m_e_tmp = m_e_inf(v_e_tmp)
        h_e_tmp = h_e + dt05 * h_e_inc
        q_e_tmp = q_e + dt05 * q_e_inc
        s_e_tmp = s_e + dt05 * s_e_inc
        q_stoch_tmp = q_stoch + dt05 * q_stoch_inc
        s_stoch_tmp = s_stoch + dt05 * s_stoch_inc

        v_i_tmp = v_i + dt05 * v_i_inc
        n_i_tmp = n_i + dt05 * n_i_inc
        m_i_tmp = m_i_inf(v_i_tmp)
        h_i_tmp = h_i + dt05 * h_i_inc
        q_i_tmp = q_i + dt05 * q_i_inc
        s_i_tmp = s_i + dt05 * s_i_inc

        v_e_inc = (0.1 * (-67 - v_e_tmp) + 80 * n_e_tmp ** 4 * (-100 - v_e_tmp)
                    + 100 * m_e_tmp ** 3 * h_e_tmp * (50 - v_e_tmp)
                    + (g_ee.T @ s_e_tmp) * (v_rev_e - v_e_tmp) + (g_ie.T @ s_i_tmp) * (v_rev_i - v_e_tmp)
                    + i_ext_e + g_stoch * s_stoch_tmp * (v_rev_e - v_e_tmp))
        n_e_inc = (n_e_inf(v_e_tmp) - n_e_tmp) / tau_n_e(v_e_tmp)
        h_e_inc = (h_e_inf(v_e_tmp) - h_e_tmp) / tau_h_e(v_e_tmp)
        q_e_inc = (1 + tanh(v_e_tmp / 10)) / 2 * (1 - q_e_tmp) / 0.1 - q_e_tmp / tau_dq_e
        s_e_inc = q_e_tmp * (1 - s_e_tmp) / tau_r_e - s_e_tmp / tau_d_e
        q_stoch_inc = -q_stoch_tmp / tau_dq_e
        s_stoch_inc = q_stoch_tmp * (1 - s_stoch_tmp) / tau_r_e - s_stoch_tmp / tau_d_e

        v_i_inc = (0.1 * (-65 - v_i_tmp) + 9 * n_i_tmp ** 4 * (-90 - v_i_tmp)
                    + 35 * m_i_tmp ** 3 * h_i_tmp * (55 - v_i_tmp)
                    + (g_ei.T @ s_e_tmp) * (v_rev_e - v_i_tmp) + (g_ii.T @ s_i_tmp) * (v_rev_i - v_i_tmp) + i_ext_i)
        n_i_inc = (n_i_inf(v_i_tmp) - n_i_tmp) / tau_n_i(v_i_tmp)
        h_i_inc = (h_i_inf(v_i_tmp) - h_i_tmp) / tau_h_i(v_i_tmp)
        q_i_inc = (1 + tanh(v_i_tmp / 10)) / 2 * (1 - q_i_tmp) / 0.1 - q_i_tmp / tau_dq_i
        s_i_inc = q_i_tmp * (1 - s_i_tmp) / tau_r_i - s_i_tmp / tau_d_i

        v_e_old, v_i_old = v_e, v_i

        v_e = v_e + dt * v_e_inc
        m_e = m_e_inf(v_e)
        h_e = h_e + dt * h_e_inc
        n_e = n_e + dt * n_e_inc
        q_e = q_e + dt * q_e_inc
        s_e = s_e + dt * s_e_inc
        q_stoch = q_stoch + dt * q_stoch_inc
        s_stoch = s_stoch + dt * s_stoch_inc

        v_i = v_i + dt * v_i_inc
        m_i = m_i_inf(v_i)
        h_i = h_i + dt * h_i_inc
        n_i = n_i + dt * n_i_inc
        q_i = q_i + dt * q_i_inc
        s_i = s_i + dt * s_i_inc

        which_e = np.where((v_e_old > -20) & (v_e <= -20))[0]
        which_i = np.where((v_i_old > -20) & (v_i <= -20))[0]
        if len(which_e) > 0:
            i_e_spikes.extend(which_e.tolist())
            t_e_spikes.extend((((-20 - v_e[which_e]) * (k - 1) * dt + (v_e_old[which_e] + 20) * k * dt)
                                / (-v_e[which_e] + v_e_old[which_e])).tolist())
        if len(which_i) > 0:
            i_i_spikes.extend(which_i.tolist())
            t_i_spikes.extend((((-20 - v_i[which_i]) * (k - 1) * dt + (v_i_old[which_i] + 20) * k * dt)
                                / (-v_i[which_i] + v_i_old[which_i])).tolist())

        # random Poisson input arrivals: an event lands on E-cell i whenever
        # u[i] < f_stoch/1000*dt, with probability f_stoch/1000*dt.
        u = rng.random(num_e)
        ind = u < f_stoch / 1000 * dt
        q_stoch[ind] = 1.

        v_one[k] = v_e[track_cell]

    t_e_spikes, i_e_spikes = np.array(t_e_spikes), np.array(i_e_spikes)
    t_i_spikes, i_i_spikes = np.array(t_i_spikes), np.array(i_i_spikes)
    return t_e_spikes, i_e_spikes, t_i_spikes, i_i_spikes, v_one, num_e, num_i, m_steps


def simulate_poisson_ping_1(t_final=500., seed=63806, track_cell=2):
    return simulate_poisson_ping(
        num_e=200, num_i=200 // 4, i_ext_e_mean=0.5, sigma_e=0.05,
        i_ext_i_mean=0.0, sigma_i=0.05, g_hat_ee=0., g_hat_ei=0.25, g_hat_ie=0.25, g_hat_ii=0.25,
        p_ee=0.5, p_ei=0.5, p_ie=0.5, p_ii=0.5,
        tau_r_e=0.5, tau_peak_e=0.5, tau_d_e=3., tau_r_i=0.5, tau_peak_i=0.5, tau_d_i=9.,
        f_stoch=60., g_stoch=0.03, t_final=t_final, seed=seed, i_init="constant", track_cell=track_cell)


def simulate_poisson_ping_2(t_final=500., seed=63806, track_cell=2):
    return simulate_poisson_ping(
        num_e=200, num_i=50, i_ext_e_mean=0.5, sigma_e=0.05,
        i_ext_i_mean=0.8, sigma_i=0.05, g_hat_ee=0., g_hat_ei=0.25, g_hat_ie=0.25, g_hat_ii=0.25,
        p_ee=0.5, p_ei=0.5, p_ie=0.5, p_ii=0.5,
        tau_r_e=0.5, tau_peak_e=0.5, tau_d_e=3., tau_r_i=0.5, tau_peak_i=0.5, tau_d_i=9.,
        f_stoch=60., g_stoch=0.03, t_final=t_final, seed=seed, i_init="wb", track_cell=track_cell)


def simulate_poisson_ping_3(t_final=500., seed=63806, track_cell=2):
    return simulate_poisson_ping(
        num_e=200, num_i=50, i_ext_e_mean=0.60, sigma_e=0.0,
        i_ext_i_mean=0.60, sigma_i=0.0, g_hat_ee=0., g_hat_ei=1.25, g_hat_ie=1.25, g_hat_ii=0.4,
        p_ee=1.0, p_ei=1.0, p_ie=1.0, p_ii=1.0,
        tau_r_e=0.3, tau_peak_e=0.3, tau_d_e=3., tau_r_i=0.3, tau_peak_i=0.3, tau_d_i=9.,
        f_stoch=40., g_stoch=0.1, t_final=t_final, seed=seed, i_init="wb", track_cell=track_cell)

In [ ]:
t_e_1, i_e_1, t_i_1, i_i_1, v_one_1, num_e_1, num_i_1, m_steps_1 = simulate_poisson_ping_1()
plot_ping_raster(t_e_1, i_e_1, t_i_1, i_i_1, num_e_1, num_i_1, 500., title="POISSON_PING_1")
plt.tight_layout()
plt.show()

`f_stoch` sets the mean Poisson event rate onto each E cell -- higher
rates drive individual E cells more often but less regularly than a
constant current would.

In [ ]:
def _poisson_ping_1_at(f_stoch, t_final=300.):
    return simulate_poisson_ping(
        num_e=200, num_i=50, i_ext_e_mean=0.5, sigma_e=0.05, i_ext_i_mean=0.0, sigma_i=0.05,
        g_hat_ee=0., g_hat_ei=0.25, g_hat_ie=0.25, g_hat_ii=0.25,
        p_ee=0.5, p_ei=0.5, p_ie=0.5, p_ii=0.5,
        f_stoch=f_stoch, g_stoch=0.03, t_final=t_final, i_init="constant")


interact(lambda f_stoch=60.0: plot_ping_raster(
    *_poisson_ping_1_at(f_stoch)[:4], 200, 50, 300., title=f"f_stoch = {f_stoch:.0f} Hz"),
    f_stoch=(20.0, 120.0, 10.0));

In [ ]:
t_e_2, i_e_2, t_i_2, i_i_2, v_one_2, num_e_2, num_i_2, m_steps_2 = simulate_poisson_ping_2()
plot_ping_raster(t_e_2, i_e_2, t_i_2, i_i_2, num_e_2, num_i_2, 500., title="POISSON_PING_2")
plt.tight_layout()
plt.show()

In [ ]:
t_e_3, i_e_3, t_i_3, i_i_3, v_one_3, num_e_3, num_i_3, m_steps_3 = simulate_poisson_ping_3()
plot_ping_raster(t_e_3, i_e_3, t_i_3, i_i_3, num_e_3, num_i_3, 500., title="POISSON_PING_3")
plt.tight_layout()
plt.show()

## POISSON_PING_3_VOLTAGE_TRACE: single E-cell voltage trace

Same simulation as `POISSON_PING_3` (`simulate_poisson_ping_3` already
returns `v_one`, the tracked E cell's voltage trace at every step);
plotted here as its own figure alongside the raster and population rate,
matching the matlab source's separate voltage-trace figure.

In [ ]:
t_e_v, i_e_v, t_i_v, i_i_v, v_one, num_e_v, num_i_v, m_steps_v = simulate_poisson_ping_3(track_cell=2)
dt = 500. / m_steps_v

fig, axes = plt.subplots(2, 1, figsize=(8, 6))
plot_ping_raster(t_e_v, i_e_v, t_i_v, i_i_v, num_e_v, num_i_v, 500., ax=axes[0],
                  title="POISSON_PING_3_VOLTAGE_TRACE")
axes[1].plot(np.arange(m_steps_v + 1) * dt, v_one, '-k', linewidth=2)
axes[1].set_xlabel('$t$ [ms]')
axes[1].set_ylabel('$v$ [mV]')
plt.tight_layout()
plt.show()